In [1]:
import json
import uuid
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
url = "https://sm.apps.dev-terra000003-ids.ocp.delta.sbrf.ru"
cert = (
    "../gigachat/published.pem",
    "../gigachat/00CA0001CLLMllm99usr.key"
)
HEADERS = {
    'accept': 'application/json',
    'Content-Type': 'application/json'
}
SOURCE_ID = "eb3669d8-7773-44d6-83a9-41e1d6f5448e"    # вставить свой source_uuid

In [3]:
# https://developers.sber.ru/docs/ru/gigachat/models/embeddings
EMBEDDER = "gigachat-3b"
EMBEDDER_SPACES = {
    "gigachat": 1024,
    "gigachat-2": 1024,
    "gigachat-gigar": 2560,
    "gigachat-3b": 2048
}
MAX_TOKENS = {
    "gigachat": 510,
    "gigachat-2": 510,
    "gigachat-gigar": 4094,
    "gigachat-3b": 4094
}

## Универсальный сценарий GigaSearch

#### GigaSearch Cookbook (Confluence):

1. [Тестирование на ИФТ](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=14408975693&src=contextnavpagetreemode)

2. [Формат взаимодействия по API](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=13630875272&src=contextnavpagetreemode)

3. [Настройка компонент пайплайна](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=15772722435&src=contextnavpagetreemode)

4. [Get Started - End2End примеры в Delta](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=13630875157&src=contextnavpagetreemode)

### 1. Поисковые запросы

In [7]:
# загрузка конфигурации поиска
with open("../configs/search_config.v.0.2.4.json", "r") as f:
    config = json.load(f)


# формирование запроса
data = {
    "request_type": "chat",
    "request": {
        "messages": [
            {
                "role": "user",
                "content": "Что такое капитализация по вкладу и зачем она нужна?"
            }
        ],
        ## Optional фльтры
        # "filters": {
        #     "operator": "and",   # "or"
        #     "elements": [
        #         {
        #             "field": "smartnote_name",
        #             "value": "Проценты по вкладам",
        #             "operator": "eq"
        #         },
        #         {
        #             "field": "breadcrumbs",
        #             "value": "*Платежи и переводы*",
        #             "operator": "wildcard"
        #         }
        #     ]
        # }
    }
}


external_id = str(uuid.uuid4())

json_data = {
    "meta": {
      "external_uuid": external_id,
      "source_uuid": SOURCE_ID
    },
    "message": data,
    "path": "/predict",
    "timeout": 300,
    "configuration": config
}

response = requests.post(
    f'{url}/sync/skill/universal_search',
    headers=HEADERS,
    json=json_data,
    cert=cert,
    verify=False
)

response.json()

{'meta': {'external_uuid': 'e3150169-0384-4ebc-9213-8cde6384a3de',
  'source_uuid': 'eb3669d8-7773-44d6-83a9-41e1d6f5448e'},
 'status': 'PROCESSING_DONE',
 'details': None,
 'result': {'response_type': 'chat',
  'response': {'messages': [{'role': 'user',
     'content': 'Что такое капитализация по вкладу и зачем она нужна?'},
    {'message_id': '737960d7-6294-48e6-b011-00d13572f990',
     'message_dt': '2026-07-06T10:00:08Z',
     'message_reason': 'llm_rag',
     'from_skill_cache': False,
     'content': '',
     'role': 'assistant',
     'status': 'complete',
     'inline_sources': [],
     'faq_sources': [{'faq_id': 'f97a4f3c-288f-42d3-90d9-99a3d2b86713',
       'relevance_score': 0.8753493,
       'retrieval_relevance_score': 0.8753493,
       'metadata': {'passage_index': 2,
        'file_name': '0d44eb8a-0bb2-4115-949c-54b9727163e7.html',
        'chapter_id': '12b659f0-d135-484e-aaba-7d459e8f935d',
        'document_id': '9b121a5a-260b-4b8e-830d-90013eabf0e8',
        'raw_text

In [11]:
from __future__ import annotations

import subprocess
import sys
import time
from typing import Any


def _ensure_opensearch_py() -> None:
    try:
        import opensearchpy  # noqa: F401
        return
    except ImportError:
        pass

    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "opensearch-py"],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0:
        return

    raise ImportError(
        "opensearch-py is missing and automatic install failed.\n"
        f"Kernel python: {sys.executable}\n"
        f"pip stdout:\n{proc.stdout}\n"
        f"pip stderr:\n{proc.stderr}\n"
        "Install manually in this kernel, then re-run the cell:\n"
        f"  {sys.executable} -m pip install opensearch-py"
    )


_ensure_opensearch_py()

from opensearchpy import OpenSearch, RequestsHttpConnection
from opensearchpy.exceptions import NotFoundError

# --- OpenSearch client (mTLS) -------------------------------------------------

client_cert = "../gigachat/published.pem"
client_key = "../gigachat/00CA0001CLLMllm99usr.key"

client = OpenSearch(
    hosts=[{
        "host": "sm.apps.dev-terra000003-ids.ocp.delta.sbrf.ru",
        "port": 443,
    }],
    http_compress=False,
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
    connection_class=RequestsHttpConnection,
    client_cert=client_cert,
    client_key=client_key,
    url_prefix="/opensearch",
)

# Default index from search_config.v.0.2.4.json; override per call if needed.
DEFAULT_INDEX_ID = (
    config.get("agent_configuration", {})
    .get("data_sources_faq", [{}])[0]
    .get("index_id", "1a0cbc8d-5df4-4bb2-abda-b4bdddfd708d_search_usooper_3")
)


def _get_path(obj: dict[str, Any], path: str) -> Any:
    cur: Any = obj
    for part in path.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return None
        cur = cur[part]
    return cur


def _pick(src: dict[str, Any], *paths: str) -> Any:
    for path in paths:
        val = _get_path(src, path)
        if val is not None:
            return val
    return None


def _hit_from_source(doc_id: str, src: dict[str, Any]) -> dict[str, Any]:
    """Map one OpenSearch _source into the tool-server neighbour shape."""
    chunk_index = _pick(src, "metadata.passage_index", "metadata.index", "passage_index", "index")
    return {
        "doc_id": doc_id,
        "title": _pick(src, "metadata.breadcrumbs", "breadcrumbs", "title") or "",
        "text": _pick(src, "metadata.raw_text", "content", "raw_text") or "",
        "file_name": _pick(src, "metadata.file_name", "file_name"),
        "index": int(chunk_index) if chunk_index is not None else None,
    }


def _term_queries(field: str, value: str) -> list[dict[str, Any]]:
    return [
        {"term": {f"{field}.keyword": value}},
        {"term": {field: value}},
    ]


def _search_same_file(
    client: OpenSearch,
    index: str,
    file_name: str,
    *,
    file_name_fields: tuple[str, ...] = ("file_name", "metadata.file_name"),
    size: int = 1000,
) -> list[dict[str, Any]]:
    """Return all chunks that share ``file_name`` in ``index``."""
    should: list[dict[str, Any]] = []
    for field in file_name_fields:
        should.extend(_term_queries(field, file_name))

    body = {
        "query": {"bool": {"should": should, "minimum_should_match": 1}},
        "size": size,
        "_source": True,
    }
    resp = client.search(index=index, body=body)
    return resp.get("hits", {}).get("hits", [])


def get_neighbours(
    client: OpenSearch,
    index: str,
    doc_id: str,
    window: int = 1,
) -> dict[str, Any]:
    """Adjacent chunks for a sliced passage — mirrors tool_server ``get_neighbours``.

  ``doc_id`` is the OpenSearch ``_id`` (same as GigaSearch ``faq_id``).
  ``index`` is the OpenSearch index name (e.g. from ``data_sources_faq[].index_id``).

  Status values match the in-repo tool server:
    * ``unknown_doc`` — doc_id not found in the index
    * ``independent_chunk`` — anchor has no file_name/index metadata
    * ``ok`` — neighbours returned (possibly empty)
    """
    t0 = time.perf_counter()
    window = max(0, int(window))

    try:
        anchor = client.get(index=index, id=doc_id)
    except NotFoundError:
        return {
            "results": [],
            "status": "unknown_doc",
            "anchor_doc_id": doc_id,
            "window": window,
            "latency_ms": int((time.perf_counter() - t0) * 1000),
        }

    anchor_src = anchor["_source"]
    file_name = _pick(anchor_src, "metadata.file_name", "file_name")
    anchor_index = _pick(
        anchor_src,
        "metadata.passage_index",
        "metadata.index",
        "passage_index",
        "index",
    )

    if file_name is None or anchor_index is None:
        return {
            "results": [],
            "status": "independent_chunk",
            "anchor_doc_id": doc_id,
            "window": window,
            "latency_ms": int((time.perf_counter() - t0) * 1000),
        }

    anchor_index = int(anchor_index)
    lo, hi = anchor_index - window, anchor_index + window

    hits = _search_same_file(client, index, str(file_name))
    neighbours: list[dict[str, Any]] = []
    for hit in hits:
        item = _hit_from_source(hit["_id"], hit["_source"])
        idx = item.get("index")
        if idx is None or idx < lo or idx > hi or idx == anchor_index:
            continue
        neighbours.append(item)

    neighbours.sort(key=lambda x: x["index"])

    return {
        "results": neighbours,
        "status": "ok",
        "anchor_doc_id": doc_id,
        "window": window,
        "latency_ms": int((time.perf_counter() - t0) * 1000),
    }


# --- example ------------------------------------------------------------------
# doc_id from a search hit above (faq_id == OpenSearch _id)
# resp = get_neighbours(client, DEFAULT_INDEX_ID, doc_id="f97a4f3c-288f-42d3-90d9-99a3d2b86713", window=1)
# resp

In [24]:
# Example: doc_id = faq_id from a search hit above
resp = get_neighbours(
    client,
    DEFAULT_INDEX_ID,
    doc_id="f97a4f3c-288f-42d3-90d9-99a3d2b86713",
    window=1,
)
resp

{'results': [{'doc_id': '8d338b96-bf29-4ccb-bc6b-5d413f44b938',
   'title': 'Общее пространство ЦКР (УСО, ТП, УУП)/New page IDP 1.2 в USOOPER',
   'text': 'Например, если сотрудник превысит установленный лимит, операция может быть отклонена банком.\nВажно помнить, что все траты подлежат обязательному контролю и отчетности согласно внутренним правилам компании и законодательству.\nПлатежи\n6.\nЧем принципиально отличаются платежное поручение и платёжная квитанция?\nПлатёжное поручение — это документ, который физическое или юридическое лицо направляет своему банку с целью произвести перечисление денежных средств другому субъекту.\nОбычно оформляется специальным бланком или через систему дистанционного банковского обслуживания.\nБанк обязан исполнить данное поручение в установленные сроки.Платёжная квитанция, напротив, является документом, подтверждающим факт совершения конкретной финансовой операции (например, оплата услуг ЖКХ).\nОна выдается самим банком плательщику сразу после завершен

### CKR eval: batch search + ndcg@16 / recall@3 / recall@5

Runs `universal_search` for each question in `data/ckr_eval/ckr_eval.jsonl` (strips `<client>` tags). Gold chunk ids are `{document_id}_{index}`; search hits are mapped via `file_name` → `document_id` from `ckr_index.jsonl`.

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from eval_.ckr_ids import ckr_chunk_id, load_file_name_to_document_id, strip_client_tags
from eval_.metrics import ndcg_at_k
from eval_.run_service_eval import recall_at_k_items

EVAL_PATH = Path("../data/ckr_eval/ckr_eval.jsonl")
INDEX_PATH = Path("../data/ckr_eval/ckr_index.jsonl")
OUT_PATH = Path("../data/processed/ckr_eval/ckr_eval_gigasearch_results.json")
KS = (3, 5, 16)
TOP_K = max(KS)
SUBSET_SIZE = None  # set e.g. 5 for a quick smoke test


def parse_hits(raw: dict) -> list[dict]:
    messages = raw["result"]["response"]["messages"]
    faq_sources = []
    for msg in reversed(messages):
        if msg.get("role") == "assistant":
            faq_sources = msg.get("faq_sources") or []
            break
    hits = []
    for item in faq_sources:
        meta = item.get("metadata") or {}
        hits.append(
            {
                "doc_id": str(item["faq_id"]),
                "file_name": meta.get("file_name"),
                "index": meta.get("passage_index"),
                "score": float(item.get("relevance_score") or 0.0),
            }
        )
    return hits[:TOP_K]


def hits_to_ranked(hits: list[dict], fn_to_doc: dict[str, str]):
    ranked_chunks: list[str] = []
    items: list[set[str]] = []
    seen: set[str] = set()
    for hit in hits:
        doc_id = hit["doc_id"]
        file_name = hit.get("file_name")
        index = hit.get("index")
        chunk_id = None
        if file_name is not None and index is not None:
            chunk_id = ckr_chunk_id(str(file_name), index, file_name_to_document_id=fn_to_doc)
        ids = {doc_id}
        if chunk_id:
            ids.add(chunk_id)
        items.append(ids)
        if chunk_id and chunk_id not in seen:
            ranked_chunks.append(chunk_id)
            seen.add(chunk_id)
    return ranked_chunks, items


def universal_search(question: str) -> dict:
    payload = {
        "meta": {"external_uuid": str(uuid.uuid4()), "source_uuid": SOURCE_ID},
        "message": {
            "request_type": "chat",
            "request": {"messages": [{"role": "user", "content": question}]},
        },
        "path": "/predict",
        "timeout": 300,
        "configuration": config,
    }
    resp = requests.post(
        f"{url}/sync/skill/universal_search",
        headers=HEADERS,
        json=payload,
        cert=cert,
        verify=False,
        timeout=320,
    )
    resp.raise_for_status()
    return resp.json()


fn_to_doc = load_file_name_to_document_id(INDEX_PATH)
rows = []
with EVAL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
if SUBSET_SIZE:
    rows = rows[:SUBSET_SIZE]

per_example = []
ndcg_sums = {k: 0.0 for k in KS}
recall_sums = {k: 0.0 for k in KS}
errors = 0

for i, row in enumerate(rows, start=1):
    question_id = row["question_id"]
    question = strip_client_tags(row["question"])
    gold = set(row.get("gold_doc_ids") or [])
    try:
        raw = universal_search(question)
        hits = parse_hits(raw)
        ranked, items = hits_to_ranked(hits, fn_to_doc)
        metrics = {f"recall@{k}": recall_at_k_items(items, gold, k) for k in KS}
        metrics.update({f"ndcg@{k}": ndcg_at_k(ranked, gold, k) for k in KS})
        for k in KS:
            recall_sums[k] += metrics[f"recall@{k}"]
            ndcg_sums[k] += metrics[f"ndcg@{k}"]
        per_example.append(
            {
                "question_id": question_id,
                "question": question,
                "gold_doc_ids": sorted(gold),
                "ranked_chunk_ids": ranked,
                **metrics,
            }
        )
        print(
            f"[{i}/{len(rows)}] {question_id}  "
            f"ndcg@16={metrics['ndcg@16']:.3f}  "
            f"recall@3={metrics['recall@3']:.0%}  recall@5={metrics['recall@5']:.0%}"
        )
    except Exception as exc:
        errors += 1
        per_example.append({"question_id": question_id, "question": question, "error": str(exc)})
        print(f"[{i}/{len(rows)}] {question_id}  ERROR: {exc}")

n = len(rows) - errors or 1
summary = {
    "n": len(rows) - errors,
    "errors": errors,
    **{f"recall@{k}": recall_sums[k] / n for k in KS},
    **{f"ndcg@{k}": ndcg_sums[k] / n for k in KS},
}
print("\n=== summary ===")
print(
    f"n={summary['n']}  errors={summary['errors']}  "
    f"ndcg@16={summary['ndcg@16']:.4f}  "
    f"recall@3={summary['recall@3']:.1%}  recall@5={summary['recall@5']:.1%}"
)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(
    json.dumps(
        {
            "eval_path": str(EVAL_PATH),
            "mode": "gigasearch_direct_notebook",
            "ks": list(KS),
            "summary": summary,
            "per_example": per_example,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"wrote {OUT_PATH}")